In [1]:
import pandas as pd
import numpy as np
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize

df = pd.read_csv('../data/processed/ab_test_clean.csv')

n_ad = (df['test_group'] == 'ad').sum()
n_psa = (df['test_group'] == 'psa').sum()
p_ad = df.loc[df['test_group'] == 'ad', 'converted'].mean()
p_psa = df.loc[df['test_group'] == 'psa', 'converted'].mean()

print('ad users: ', n_ad, '| psa users: ', n_psa)
print("ad rate:", round(p_ad * 100, 2), "% | psa rate:", round(p_psa * 100, 2), "%")

ad users:  564577 | psa users:  23524
ad rate: 2.55 % | psa rate: 1.79 %


In [2]:
power_calc = NormalIndPower()

h = power_calc.solve_power(effect_size=None, nobs1=n_psa, alpha=0.05,
                            power=0.8, ratio=n_ad/n_psa)

p_needed = np.sin((h + 2 * np.arcsin(np.sqrt(p_psa))) / 2) ** 2

print('PSA rate: ', round(p_psa * 100, 2), '%')
print('ad rate needed for 80% power: ', round(p_needed * 100, 2), '%')
print('minimum detectable lift: ', round((p_needed - p_psa) * 100, 2), 'percentage points')

PSA rate:  1.79 %
ad rate needed for 80% power:  2.04 %
minimum detectable lift:  0.26 percentage points


In [3]:
observed_es = proportion_effectsize(p_ad, p_psa)
observed_power = power_calc.solve_power(effect_size=observed_es, nobs1=n_psa,
                                        alpha=0.05, ratio=n_ad/n_psa)

print('power at the observed effect: ', round(observed_power, 4))

power at the observed effect:  1.0


In [4]:
rows = []
for lift in [0.002, 0.003, 0.005, 0.0077]:
    es = proportion_effectsize(p_psa + lift, p_psa)
    n = power_calc.solve_power(effect_size=es, alpha=0.05, power=0.8, ratio=1)
    rows.append({"lift_pp": lift * 100, "users_per_group": int(np.ceil(n))})

plan = pd.DataFrame(rows)
plan["total_users"] = plan["users_per_group"] * 2
plan

,lift_pp,users_per_group,total_users
0,0.20,72547,145094
1,0.30,33055,66110
2,0.50,12474,24948
3,0.77,5578,11156
